# Exploring a Personal Genome at 30x Depth

**University of Westminster | Whole-Genome Sequencing Workshop | 2026**

**Dr Manuel Corpas** | Senior Lecturer in Genomics, AI & Health Data Science

---

In this tutorial you will:

1. Load and explore a **real 30x whole-genome sequence** (CC0 public domain)
2. Compute QC metrics: Ti/Tv ratio, Het/Hom ratio, variant counts
3. Explore **structural variants** (deletions, duplications, inversions, insertions)
4. Extract and compare **pharmacogenomic variants** from WGS vs SNP chip
5. Annotate selected variants with VEP, ClinVar, and gnomAD

Everything runs in this Colab notebook. No local installation needed.

> **About this dataset**: This is the 30x Illumina whole-genome sequence of Manuel Corpas, published under CC0 on Zenodo ([doi:10.5281/zenodo.19297389](https://doi.org/10.5281/zenodo.19297389)). It extends the Corpasome, one of the first fully open personal genomes.

> **Research and education only**: This dataset must not be used for clinical decision-making. ClawBio is not a medical device.

## Step 0: Setup

Install ClawBio and dependencies.

In [ ]:
%%bash
# Clone ClawBio repository
if [ ! -d "ClawBio" ]; then
    git clone --depth 1 https://github.com/ClawBio/ClawBio.git
fi

# Install dependencies
pip install -q pysam requests matplotlib pandas

In [ ]:
import sys
import os

sys.path.insert(0, 'ClawBio')
os.chdir('ClawBio')

print('ClawBio loaded successfully')
print(f'Skills available: {len(os.listdir("skills"))}')

# Check that the 30x WGS subsets are present
from pathlib import Path
corpas_dir = Path('corpas-30x')
subsets = list((corpas_dir / 'subsets').glob('*.vcf.gz'))
print(f'WGS subsets available: {len(subsets)}')
for s in sorted(subsets):
    size_kb = s.stat().st_size / 1024
    print(f'  {s.name}: {size_kb:.0f} KB')

## Step 1: Explore the 30x WGS Data

The Corpasome 30x WGS contains variant calls across all chromosomes. We will start with the chr20 subset, which is small enough for interactive analysis.

In [ ]:
import gzip
import json
from pathlib import Path
from collections import Counter

# Load the QC baselines (pre-computed from the full genome)
baselines_path = Path('corpas-30x/baselines/qc_summary.json')
with open(baselines_path) as f:
    baselines = json.load(f)

print('=== Corpas 30x WGS: Full Genome Summary ===')
print(f'Genome build: {baselines["genome_build"]}')
print()

snp_stats = baselines.get('snp_stats', {})
indel_stats = baselines.get('indel_stats', {})
print(f'Total SNPs:    {snp_stats.get("total_snps", "?"):>12,}')
print(f'Total indels:  {indel_stats.get("total_indels", "?"):>12,}')
print(f'Ti/Tv ratio:   {snp_stats.get("ts_tv_ratio", "?"):>12}')
print(f'Het/Hom ratio: {snp_stats.get("het_hom_ratio", "?"):>12}')
print()

sv_counts = baselines.get('sv_counts', {})
sv_total = baselines.get('sv_total', 0)
cnv_total = baselines.get('cnv_total', 0)
print(f'Structural variants: {sv_total:,}')
for svtype, count in sorted(sv_counts.items(), key=lambda x: -x[1]):
    print(f'  {svtype:>4}: {count:>6,}')
print(f'Copy number variants: {cnv_total:,}')

### Understanding QC Metrics

- **Ti/Tv ratio** (transitions/transversions): Expected ~2.0 for whole-genome SNPs. Values significantly below 2.0 suggest contamination or sequencing errors. Our value: **2.03** (excellent).

- **Het/Hom ratio** (heterozygous/homozygous alternate): Expected ~1.5-1.7 for a single outbred individual. Values outside this range may indicate sample contamination or consanguinity. Our value: **1.63** (normal).

- **Variant count**: ~3.5-4.5 million SNPs expected for a European genome at 30x coverage. Our count: **3,716,648** (within expected range).

**Question for students**: Why is the Ti/Tv ratio expected to be ~2.0? (Transitions are chemically more likely than transversions due to the molecular structure of purines and pyrimidines.)

In [ ]:
import subprocess

# Count variants in the chr20 subset
chr20_vcf = Path('corpas-30x/subsets/chr20_snps_indels.vcf.gz')

result = subprocess.run(
    ['python', '-c', f'''
import gzip
count = 0
with gzip.open("{chr20_vcf}", "rt") as f:
    for line in f:
        if not line.startswith("#"):
            count += 1
print(count)
'''],
    capture_output=True, text=True
)
chr20_count = int(result.stdout.strip()) if result.stdout.strip() else 0
print(f'Chromosome 20 variants: {chr20_count:,}')
print()
print('Preview (first 5 data lines):')
print('CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER')
print('-' * 60)

with gzip.open(chr20_vcf, 'rt') as f:
    shown = 0
    for line in f:
        if not line.startswith('#'):
            fields = line.strip().split('\t')
            print(f'{fields[0]}\t{fields[1]}\t{fields[2]}\t{fields[3]}\t{fields[4]}\t{fields[5]}\t{fields[6]}')
            shown += 1
            if shown >= 5:
                break

## Step 2: Explore Structural Variants

Structural variants (SVs) are genomic rearrangements larger than 50 base pairs. They include deletions, duplications, inversions, insertions, and breakends (translocations). SVs are invisible to SNP arrays but captured by WGS.

In [ ]:
import gzip
from collections import Counter

sv_vcf = Path('corpas-30x/subsets/sv_calls.vcf.gz')

sv_types = Counter()
sv_sizes = []
sv_chroms = Counter()

with gzip.open(sv_vcf, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        chrom = fields[0]
        info = fields[7]

        # Extract SVTYPE
        svtype = None
        for part in info.split(';'):
            if part.startswith('SVTYPE='):
                svtype = part.split('=')[1]
                break

        if svtype:
            sv_types[svtype] += 1
            sv_chroms[chrom] += 1

            # Extract size (END - POS) for non-BND types
            if svtype != 'BND':
                end = None
                for part in info.split(';'):
                    if part.startswith('END='):
                        end = int(part.split('=')[1])
                        break
                if end:
                    size = abs(end - int(fields[1]))
                    sv_sizes.append((svtype, size))

print('=== Structural Variant Summary ===')
print(f'Total SVs: {sum(sv_types.values()):,}')
print()
for svtype, count in sv_types.most_common():
    print(f'  {svtype:>4}: {count:>6,}')

print()
print('=== Size Distribution (non-BND) ===')
if sv_sizes:
    sizes_only = [s for _, s in sv_sizes]
    print(f'  Smallest: {min(sizes_only):,} bp')
    print(f'  Largest:  {max(sizes_only):,} bp')
    print(f'  Median:   {sorted(sizes_only)[len(sizes_only)//2]:,} bp')

    # Size bins
    bins = [(50, 500, '50-500bp'), (500, 5000, '500bp-5kb'),
            (5000, 50000, '5-50kb'), (50000, 500000, '50-500kb'),
            (500000, float('inf'), '>500kb')]
    print()
    print('  Size distribution:')
    for lo, hi, label in bins:
        n = sum(1 for _, s in sv_sizes if lo <= s < hi)
        bar = '#' * (n // 20)
        print(f'    {label:>10}: {n:>5,} {bar}')

### What do these SV types mean?

| Type | Full name | Biological meaning |
|------|-----------|-------------------|
| **DEL** | Deletion | A segment of DNA is missing. Can disrupt genes if it overlaps coding regions. |
| **DUP** | Duplication | A segment is copied. Gene duplications can alter dosage (e.g., CYP2D6 copy number). |
| **INV** | Inversion | A segment is flipped in orientation. Can disrupt genes at breakpoints. |
| **INS** | Insertion | New DNA is inserted. Mobile element insertions (Alu, LINE) are common. |
| **BND** | Breakend | A translocation or complex rearrangement. Two distant genomic positions are joined. |

**Key insight**: Deletions dominate (5,854 in this genome). This is expected: deletions are easier to detect than other SV types, and many represent common population polymorphisms.

**Question for students**: The largest SV in this genome spans how many bases? What gene(s) might it overlap?

## Step 3: Pharmacogenomic Variants from WGS

SNP arrays test pre-selected positions. WGS captures everything. Let us compare what each platform finds at pharmacogenomic loci.

In [ ]:
import gzip

# Load PGx variants from WGS
pgx_vcf = Path('corpas-30x/subsets/pgx_loci.vcf.gz')
pgx_23andme = Path('skills/pharmgx-reporter/demo_patient.txt')

# WGS PGx variants
wgs_variants = []
with gzip.open(pgx_vcf, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        wgs_variants.append({
            'chrom': fields[0],
            'pos': fields[1],
            'ref': fields[3],
            'alt': fields[4],
            'qual': fields[5],
            'filter': fields[6],
        })

# 23andMe PGx variants
chip_variants = []
with open(pgx_23andme) as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.strip().split('\t')
        if len(parts) >= 4:
            chip_variants.append({
                'rsid': parts[0],
                'chrom': parts[1],
                'pos': parts[2],
                'genotype': parts[3],
            })

print('=== Pharmacogenomic Variant Comparison ===')
print()
print(f'23andMe SNP chip: {len(chip_variants)} PGx variants (pre-selected rsIDs)')
print(f'30x WGS:          {len(wgs_variants)} variants at PGx loci')
print()
print('--- 23andMe PGx variants ---')
for v in chip_variants[:10]:
    print(f'  {v["rsid"]:>14}  chr{v["chrom"]}:{v["pos"]}  {v["genotype"]}')
if len(chip_variants) > 10:
    print(f'  ... and {len(chip_variants) - 10} more')

print()
print('--- WGS variants at PGx loci ---')
if wgs_variants:
    for v in wgs_variants:
        gt_field = ''
        # Try to extract genotype from FORMAT/SAMPLE columns
        print(f'  {v["chrom"]}:{v["pos"]}  {v["ref"]}>{v["alt"]}  QUAL={v["qual"]}  {v["filter"]}')
else:
    print('  No variant calls at these positions (all reference-homozygous)')

print()
print('Key insight: The SNP chip reports genotypes at 21 pre-selected positions,')
print('including reference-homozygous calls. WGS only reports positions where the')
print('individual differs from the reference. Positions with no WGS variant call')
print('are homozygous reference, which is itself informative (normal function).')

## Step 4: Annotate Selected Variants

Let us run ClawBio's variant-annotation skill on a small set of chr20 variants to see how the annotation pipeline works with WGS data.

In [ ]:
%%bash
# Extract the first 20 variants from chr20 for annotation
# (Keep it small for the free VEP API)

python -c "
import gzip
from pathlib import Path

vcf_in = Path('corpas-30x/subsets/chr20_snps_indels.vcf.gz')
vcf_out = Path('output/chr20_sample.vcf')
vcf_out.parent.mkdir(parents=True, exist_ok=True)

with gzip.open(vcf_in, 'rt') as fin, open(vcf_out, 'w') as fout:
    count = 0
    for line in fin:
        if line.startswith('#'):
            fout.write(line)
            continue
        # Only take PASS variants
        fields = line.strip().split('\\t')
        if fields[6] == 'PASS' or fields[6] == '.':
            fout.write(line)
            count += 1
            if count >= 20:
                break
    print(f'Extracted {count} PASS variants from chr20')
"

# Run variant annotation
python skills/variant-annotation/variant_annotation.py \
    --input output/chr20_sample.vcf \
    --output output/chr20_annotation \
    --assembly GRCh37 2>&1 || echo 'Annotation complete (check output above)'

In [ ]:
from pathlib import Path

report_path = Path('output/chr20_annotation/report.md')
if report_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(report_path.read_text()))
else:
    print('Report not found. Listing output files:')
    ann_dir = Path('output/chr20_annotation')
    if ann_dir.exists():
        for f in sorted(ann_dir.rglob('*')):
            if f.is_file():
                print(f'  {f}')
    else:
        print('  Output directory not created. Check errors above.')

## Step 5: Exercises

### Exercise 5a: Compare WGS and SNP chip pharmacogenomics

The 23andMe SNP chip found 21 pharmacogenomic variants (including reference-homozygous calls). The WGS found 5 variant calls at the same positions.

1. Why does the chip report more "variants" than the WGS? (Hint: the chip reports genotype at every tested position, including reference calls. WGS only outputs positions that differ from reference.)
2. Which platform gives you more confidence in the result? Why?
3. Can you think of a pharmacogene where WGS would find something the chip cannot? (Hint: CYP2D6 gene deletions and duplications.)

### Exercise 5b: Investigate the structural variants

1. Find the largest deletion (DEL) in the SV calls. What is its size? What chromosome is it on?
2. Does it overlap any known genes? (Use the UCSC Genome Browser at genome.ucsc.edu with the GRCh37/hg19 assembly.)
3. Are large deletions more common on some chromosomes than others?

### Exercise 5c: Novel variant discovery

1. How many of the chr20 variants have no rsID (ID column is ".")?
2. What does it mean when a variant has no rsID? (It may be novel, or simply not yet catalogued in dbSNP.)
3. If you found a novel missense variant in a disease gene, what steps would you take to determine if it is pathogenic?

## Key Concepts Covered

| Concept | SNP Array | 30x WGS |
|---------|-----------|--------|
| Variant discovery | Pre-selected positions only | Genome-wide, unbiased |
| Structural variants | Not detectable | DEL, DUP, INV, INS, BND |
| Copy number variants | Limited (intensity-based) | Direct detection from read depth |
| Novel variants | Cannot find (only tests known positions) | Discovers new variants |
| Pharmacogenomics | Tests specific rsIDs | Full gene coverage including structural changes |
| QC metrics | N/A | Ti/Tv, Het/Hom, coverage depth |
| Cost (2026) | ~$100 | ~$300-500 |

**The key insight**: SNP arrays answer pre-defined questions. WGS lets you ask questions you did not know to ask. Both have their place: arrays for screening, WGS for comprehensive analysis.

## Further Reading

- ClawBio: [github.com/ClawBio/ClawBio](https://github.com/ClawBio/ClawBio)
- Zenodo dataset: [doi:10.5281/zenodo.19297389](https://doi.org/10.5281/zenodo.19297389)
- Corpasome paper: [doi:10.1186/1751-0473-8-13](https://link.springer.com/article/10.1186/1751-0473-8-13)
- Variant interpretation workshop: [workshop-variant-interpretation.html](https://clawbio.ai/workshop-variant-interpretation.html)
- ACMG Standards: Richards et al. (2015) *Genetics in Medicine* 17(5):405-24
- CPIC Guidelines: [cpicpgx.org](https://cpicpgx.org)
- Ensembl VEP: [ensembl.org/vep](https://www.ensembl.org/info/docs/tools/vep/index.html)
- gnomAD: [gnomad.broadinstitute.org](https://gnomad.broadinstitute.org/)
- UCSC Genome Browser: [genome.ucsc.edu](https://genome.ucsc.edu)

---

*This tutorial is part of ClawBio's open educational resources. Licensed under MIT.*
*Dataset: CC0 1.0 Universal. You may use this data freely for any purpose without restriction.*